# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [5]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [6]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [12]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('../data/airbnb_london.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe().round(1))

# ============================================================================
# TASK 1: HISTOGRAM - Price by Room Type (Entire home/apt vs Private room)
# ============================================================================

print(f"\n\n{'='*70}")
print("TASK 1: Histogram - Price Distribution by Room Type")
print(f"{'='*70}")

# Cap price at 95th percentile
price_95 = df['price'].quantile(0.95)
print(f"\n95th percentile price: £{price_95:.2f}")

df_cap = df[df['price'] <= price_95].copy()
print(f"Listings after capping: {len(df_cap)} (removed {len(df) - len(df_cap)} outliers)")

# Filter for Entire home/apt and Private room
df_filtered = df_cap[df_cap['room_type'].isin(['Entire home/apt', 'Private room'])].copy()

print(f"\nRoom type counts:")
print(df_filtered['room_type'].value_counts())

# Calculate medians for each room type
median_entire = df_filtered[df_filtered['room_type'] == 'Entire home/apt']['price'].median()
median_private = df_filtered[df_filtered['room_type'] == 'Private room']['price'].median()

print(f"\nMedian prices:")
print(f"  Entire home/apt: £{median_entire:.2f}")
print(f"  Private room: £{median_private:.2f}")

# Create overlapping histogram using Plotly Express
fig1 = px.histogram(
    df_filtered,
    x='price',
    color='room_type',
    barmode='overlay',
    nbins=40,
    title='<b>Task 1 — Histogram: Price by Room Type (Overlapping Distributions)</b><br>' +
          f'<sub>Entire homes (median £{median_entire:.0f}) command {median_entire/median_private:.1f}x higher prices than private rooms (median £{median_private:.0f})</sub>',
    labels={'price': 'Price (£)', 'count': 'Frequency', 'room_type': 'Room Type'},
    color_discrete_map={'Entire home/apt': '#e74c3c', 'Private room': '#3498db'},
    template='plotly_white'
)

# Update traces for opacity
fig1.update_traces(opacity=0.6)

# Add vertical lines for medians
fig1.add_vline(
    x=median_entire,
    line_dash="dash",
    line_color="#e74c3c",
    line_width=2.5,
    annotation_text=f"<b>Median Entire: £{median_entire:.0f}</b>",
    annotation_position="top right",
    annotation_font_size=11,
    annotation_font_color="#e74c3c"
)

fig1.add_vline(
    x=median_private,
    line_dash="dash",
    line_color="#3498db",
    line_width=2.5,
    annotation_text=f"<b>Median Private: £{median_private:.0f}</b>",
    annotation_position="bottom right",
    annotation_font_size=11,
    annotation_font_color="#3498db"
)

fig1.update_layout(
    xaxis_title='Price (£)',
    yaxis_title='Frequency',
    hovermode='x unified',
    width=1200,
    height=550,
    font=dict(size=12),
    legend=dict(x=0.99, y=0.99, xanchor='right', yanchor='top', bgcolor='rgba(255,255,255,0.8)'),
    title_font_size=16
)

fig1.show()
fig1.write_html('task1_price_histogram.html')
print("\n✅ Task 1 visualization saved as 'task1_price_histogram.html'")



Dataset shape: (2500, 7)

First few rows:
            neighbourhood        room_type   price  minimum_nights  \
0             Hammersmith  Entire home/apt  148.55               6   
1  Kensington and Chelsea     Private room  393.56              29   
2                  Camden  Entire home/apt  197.64              19   
3               Southwark     Private room   70.47               6   
4             Hammersmith     Private room   96.58              18   

   number_of_reviews  availability_365  reviews_per_month  
0                158               302               3.75  
1                 46               323               2.56  
2                242               291               2.52  
3                199                22               1.24  
4                110               103               4.11  

Data types:
neighbourhood         object
room_type             object
price                float64
minimum_nights         int64
number_of_reviews      int64
availability_365   


✅ Task 1 visualization saved as 'task1_price_histogram.html'


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('../data/airbnb_london.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

# ============================================================================
# TASK 2: BOX PLOT - Listing Activity by Neighbourhood (HORIZONTAL)
# ============================================================================

print(f"\n{'='*70}")
print("TASK 2: Box Plot - Listing Activity by Neighbourhood")
print(f"{'='*70}")

# Filter out inactive listings (reviews_per_month > 0)
df_active = df[df['reviews_per_month'] > 0].copy()
print(f"\nActive listings (reviews_per_month > 0): {len(df_active)} out of {len(df)}")
print(f"Number of neighbourhoods: {df_active['neighbourhood'].nunique()}")

# Calculate median for each neighbourhood to sort
neighbourhood_medians = df_active.groupby('neighbourhood')['reviews_per_month'].median().sort_values(ascending=False)

print(f"\nTop 10 neighbourhoods by median reviews/month:")
print(neighbourhood_medians.head(10))

# Get top 2 neighbourhoods (most active)
top_two = neighbourhood_medians.head(2).index.tolist()
print(f"\n🏆 Top 2 most active neighbourhoods: {top_two[0]}, {top_two[1]}")

# Create a category for coloring top 2
df_active['is_top_two'] = df_active['neighbourhood'].apply(lambda x: 'Top 2 Active' if x in top_two else 'Other')

# Sort neighbourhoods by median for proper display order
neighbourhood_order = neighbourhood_medians.index.tolist()

# Create horizontal box plot using px.box
fig = px.box(
    df_active,
    y='neighbourhood',
    x='reviews_per_month',
    color='is_top_two',
    orientation='h',
    category_orders={'neighbourhood': neighbourhood_order},
    points='outliers',
    title='<b>Task 2 — Box Plot: Listing Activity by Borough</b><br>' +
          f'<sub><span style="color:#636EFA"><b>{top_two[0]}</b></span> and <span style="color:#636EFA"><b>{top_two[1]}</b></span> are the two most active neighbourhoods</sub>',
    labels={'reviews_per_month': 'Reviews per Month', 'neighbourhood': 'Neighbourhood', 'is_top_two': 'Status'},
    color_discrete_map={'Top 2 Active': '#27ae60', 'Other': '#95a5a6'},
    template='plotly_white'
)

# Update layout for better visualization
fig.update_layout(
    xaxis_title='Reviews per Month',
    yaxis_title='Neighbourhood',
    width=1200,
    height=1000,
    font=dict(size=11),
    hovermode='closest',
    yaxis=dict(
        tickfont=dict(size=10)
    ),
    xaxis=dict(tickfont=dict(size=11)),
    legend=dict(
        title='Status',
        x=0.99,
        y=0.99,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.8)'
    ),
    title_font_size=16
)

# Update traces
fig.update_traces(
    boxmean='sd',  # Show mean and standard deviation
    marker=dict(size=6),
    line=dict(width=1.5)
)

fig.show()
fig.write_html('task2_box_plot.html')
print("\n✅ Task 2 visualization saved as 'task2_box_plot.html'")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print(f"\n{'='*70}")
print("SUMMARY STATISTICS - TASK 2")
print(f"{'='*70}")

neighbourhood_stats = df_active.groupby('neighbourhood').agg({
    'reviews_per_month': ['median', 'mean', 'std', 'min', 'max', 'count']
}).round(2)

neighbourhood_stats.columns = ['median', 'mean', 'std', 'min', 'max', 'count']
neighbourhood_stats = neighbourhood_stats.sort_values('median', ascending=False)

print(f"\nTop 10 Neighbourhoods by Listing Activity:")
print(f"\n{'Neighbourhood':<30} {'Median':<12} {'Mean':<12} {'Std Dev':<12} {'Min':<10} {'Max':<10} {'Listings':<10}")
print(f"{'-'*95}")

for idx, (neighbourhood, row) in enumerate(neighbourhood_stats.head(10).iterrows()):
    marker = "🏆 " if neighbourhood in top_two else "   "
    print(f"{marker}{neighbourhood:<27} {row['median']:<11.2f} {row['mean']:<11.2f} {row['std']:<11.2f} {row['min']:<9.2f} {row['max']:<9.2f} {int(row['count']):<9}")

print(f"\n✅ Complete!")

Dataset shape: (2500, 7)

First few rows:
            neighbourhood        room_type   price  minimum_nights  \
0             Hammersmith  Entire home/apt  148.55               6   
1  Kensington and Chelsea     Private room  393.56              29   
2                  Camden  Entire home/apt  197.64              19   
3               Southwark     Private room   70.47               6   
4             Hammersmith     Private room   96.58              18   

   number_of_reviews  availability_365  reviews_per_month  
0                158               302               3.75  
1                 46               323               2.56  
2                242               291               2.52  
3                199                22               1.24  
4                110               103               4.11  

TASK 2: Box Plot - Listing Activity by Neighbourhood

Active listings (reviews_per_month > 0): 2495 out of 2500
Number of neighbourhoods: 10

Top 10 neighbourhoods by median re


✅ Task 2 visualization saved as 'task2_box_plot.html'

SUMMARY STATISTICS - TASK 2

Top 10 Neighbourhoods by Listing Activity:

Neighbourhood                  Median       Mean         Std Dev      Min        Max        Listings  
-----------------------------------------------------------------------------------------------
🏆 Tower Hamlets               1.75        2.24        2.03        0.04      12.90     259      
🏆 Hackney                     1.66        2.18        1.98        0.01      12.08     252      
   Camden                      1.58        2.16        1.92        0.02      9.79      246      
   Southwark                   1.52        2.14        2.08        0.01      15.07     270      
   Westminster                 1.39        1.95        1.76        0.01      7.97      260      
   Lambeth                     1.31        1.94        2.03        0.02      11.70     250      
   Hammersmith                 1.28        2.10        2.31        0.01      14.63     245  